# Direct Generation Baseline for Omnilex Legal Retrieval

This notebook implements a **direct generation approach** where we prompt a local LLM to generate Swiss legal citations based on the query.

## Approach
1. Load a local LLM (GGUF format via llama-cpp-python)
2. For each query, prompt the LLM to directly generate relevant citations
3. Parse and normalize the generated citations
4. Create submission file

## Requirements
- llama-cpp-python (optional - runs in mock mode if not installed)
- A GGUF model file (optional - uses mock responses for demonstration if not available)
- pandas, tqdm

**Note**: This notebook runs in "mock mode" if llama-cpp-python is not installed or no model file is available, generating placeholder citations for demonstration purposes.

## 1. Setup & Configuration

In [1]:
# === INSTALL DEPENDENCIES ===
!pip install -q llama-cpp-python --prefer-binary \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
!pip install -q pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB 543.3 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.4 MB/s eta 0:00:00


In [12]:
import os
import sys
from pathlib import Path

# === CONFIGURATION ===
# Choose which dataset to run on: "val" or "test"
DATASET_MODE = "test"  # Change to "test" for final submission

# Detect environment
KAGGLE_ENV = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if KAGGLE_ENV:
    # Kaggle paths (matches competition + model dataset structure)
    DATA_PATH = Path("/kaggle/input/competitions/llm-agentic-legal-information-retrieval")
    MODEL_PATH = Path("/kaggle/input/datasets/charan1996/mistral-7b-gguf")
    OUTPUT_PATH = Path("/kaggle/working")
else:
    # Local development paths
    REPO_ROOT = Path(".").resolve().parent
    DATA_PATH = REPO_ROOT / "data"
    MODEL_PATH = REPO_ROOT / "models"
    OUTPUT_PATH = REPO_ROOT / "output"

# Derived paths based on DATASET_MODE
QUERY_FILE = DATA_PATH / f"{DATASET_MODE}.csv"
IS_VALIDATION_MODE = DATASET_MODE == "val"

# Create output directory
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print(f"Environment: {'Kaggle' if KAGGLE_ENV else 'Local'}")
print(f"Dataset mode: {DATASET_MODE}")
print(f"Query file: {QUERY_FILE}")
print(f"Validation mode: {IS_VALIDATION_MODE}")
print(f"Data path: {DATA_PATH}")
print(f"Model path: {MODEL_PATH}")
print(f"Output path: {OUTPUT_PATH}")

Environment: Kaggle
Dataset mode: test
Query file: /kaggle/input/competitions/llm-agentic-legal-information-retrieval/test.csv
Validation mode: False
Data path: /kaggle/input/competitions/llm-agentic-legal-information-retrieval
Model path: /kaggle/input/datasets/charan1996/mistral-7b-gguf
Output path: /kaggle/working


In [13]:
# Configuration
CONFIG = {
    # Model settings
    "model_file": "mistral-7b-instruct-v0.2.Q4_K_M.gguf",
    "n_ctx": 4096,         # Context window
    "n_threads": 8,        # CPU threads
    "n_gpu_layers": -1,    # GPU layers (-1 = offload all layers to GPU)
    
    # Generation settings
    "max_tokens": 512,
    "temperature": 0.0,    # Deterministic for consistency
}

## 2. Load Local LLM

In [14]:
from llama_cpp import Llama

# Find model file
model_file = MODEL_PATH / CONFIG["model_file"]

if not model_file.exists():
    # Try to find any GGUF file
    gguf_files = list(MODEL_PATH.glob("*.gguf")) + list(MODEL_PATH.rglob("*.gguf"))
    if gguf_files:
        model_file = gguf_files[0]
    else:
        raise FileNotFoundError(f"No GGUF model found in {MODEL_PATH}")

print(f"Loading model: {model_file}")
llm = Llama(
    model_path=str(model_file),
    n_ctx=CONFIG["n_ctx"],
    n_threads=CONFIG["n_threads"],
    n_gpu_layers=CONFIG["n_gpu_layers"],
    verbose=False,
)
print(f"Model loaded successfully on GPU (all layers offloaded)")


Loading model: /kaggle/input/datasets/charan1996/mistral-7b-gguf/mistral-7b-instruct-v0.2.Q4_K_M.gguf
Model loaded successfully on GPU (all layers offloaded)


## 3. Define Generation Prompt

In [15]:
SYSTEM_PROMPT = """You are a Swiss legal citation expert. Output ONLY a Python list of citations.

CITATION FORMATS:
- Federal laws: "Art. X ABBREV" where ABBREV is ZGB, OR, StGB, BV, etc.
- Court decisions: "BGE X Y Z" or "BGE X Y Z E. N" with consideration number

OUTPUT FORMAT: Python list like ["citation1", "citation2", ...]

EXAMPLES:

Query: What are the requirements for a valid contract under Swiss law?
["Art. 1 OR", "Art. 11 OR", "Art. 12 OR", "BGE 119 II 449 E. 2", "BGE 127 III 248 E. 3.1"]

Query: When can a marriage be annulled in Switzerland?
["Art. 104 ZGB", "Art. 105 ZGB", "Art. 106 ZGB", "BGE 121 III 38 E. 2b"]

Query: What constitutes negligent homicide under Swiss criminal law?
["Art. 117 StGB", "Art. 12 StGB", "BGE 116 IV 306 E. 1a"]

Query: What are the grounds for divorce in Swiss law?
["Art. 111 ZGB", "Art. 112 ZGB", "Art. 114 ZGB", "Art. 115 ZGB", "BGE 130 III 585 E. 2.1"]

Query: How is inheritance distributed under Swiss law?
["Art. 457 ZGB", "Art. 462 ZGB", "Art. 471 ZGB", "BGE 132 III 305 E. 3.2"]

Now answer:"""

# Common Swiss law abbreviations for regex matching
LAW_ABBREVS = (
    "ZGB|OR|StGB|BV|SchKG|ZPO|StPO|BGG|VwVG|IPRG|KG|DSG|MSchG|URG|PatG|"
    "DesG|UWG|PrSG|FINMAG|BankG|VAG|KAG|GwG|BEHG|FinfraG|FIDLEG|FINIG|"
    "ATSG|AHV|IV|EO|ALV|KVG|UVG|BVG|ArG|GlG|USG|RPG|WaG|JSG|TSchG|"
    "LwG|PBG|EBG|SVG|LFG|SebG|SpG|BoeB|EMRK|SR|AS|BBl|ParlG|RVOG|RVOV|"
    "MG|BPG|BPV|VBGÖ|VDSG|MWSTG|DBG|StHG|VStG|StG|ZG|CO|CP|CC|CPC|CPP|"
    "LEtr|LAsi|LN|LDIP|LCart|LDA|LPM|LBI|LDes|LCD|LFINMA|LB|LSA|LPCC|"
    "LBA|LBVM|LIMF|LSFin|LEFin|LAVS|LAI|LAPG|LACI|LAMal|LAA|LPP|LTr|"
    "LEg|LPD|LPE|LAT|LFo|LChP|LPN|LAgr|LTV|LCdF|LNA|LPTh|LTAF|LTF"
)


def extract_citations(raw_output: str) -> list[str]:
    """Extract citations from raw LLM output using regex patterns."""
    import re

    citations = []

    # Pattern for BGE citations: BGE 141 II 345 E. 3.2
    # Matches: BGE + volume + part (roman) + page + optional consideration
    bge_pattern = r'BGE\s+(\d+)\s+([IVX]+[a-z]?)\s+(\d+)(?:\s+E\.\s*([\d.a-z/]+))?'
    for match in re.finditer(bge_pattern, raw_output):
        vol, part, page, consid = match.groups()
        if consid:
            citations.append(f"BGE {vol} {part} {page} E. {consid}")
        else:
            citations.append(f"BGE {vol} {part} {page}")

    # Pattern for Art. citations with Abs./lit./Ziff.
    # Matches: Art. 221 Abs. 1 lit. b StPO, Art. 364 Abs. 1 OR, Art. 1 ZGB
    art_pattern = rf'Art\.?\s*(\d+[a-z]?)(?:\s+(Abs\.?\s*\d+))?(?:\s+(lit\.?\s*[a-z]))?(?:\s+(Ziff\.?\s*\d+))?\s+({LAW_ABBREVS})\b'
    for match in re.finditer(art_pattern, raw_output, re.IGNORECASE):
        art_num, abs_part, lit_part, ziff_part, abbrev = match.groups()
        parts = [f"Art. {art_num}"]
        if abs_part:
            # Normalize "Abs1" or "Abs 1" to "Abs. 1"
            abs_normalized = re.sub(r'Abs\.?\s*', 'Abs. ', abs_part)
            parts.append(abs_normalized.strip())
        if lit_part:
            lit_normalized = re.sub(r'lit\.?\s*', 'lit. ', lit_part)
            parts.append(lit_normalized.strip())
        if ziff_part:
            ziff_normalized = re.sub(r'Ziff\.?\s*', 'Ziff. ', ziff_part)
            parts.append(ziff_normalized.strip())
        parts.append(abbrev.upper())
        citations.append(" ".join(parts))

    # Remove duplicates while preserving order
    seen = set()
    unique = []
    for c in citations:
        if c not in seen:
            seen.add(c)
            unique.append(c)

    return unique


def generate_citations(query: str) -> list[str]:
    """Generate citations using direct LLM prompting."""
    if llm is None:
        raise RuntimeError("LLM not loaded - please ensure model is available")

    prompt = f"[INST] {SYSTEM_PROMPT}\n\nQuery: {query} [/INST]"

    response = llm(
        prompt,
        max_tokens=CONFIG["max_tokens"],
        temperature=CONFIG["temperature"],
        stop=["[INST]", "</s>", "Query:", "\n\n"],
    )

    raw_output: str = response["choices"][0]["text"].strip()  # type: ignore[index]
    print(f"Raw output: {raw_output}\n")

    # Parse Python list format first
    import re
    import ast
    
    citations = []
    
    # Try to parse as Python list
    try:
        list_match = re.search(r'\[.*?\]', raw_output, re.DOTALL)
        if list_match:
            parsed = ast.literal_eval(list_match.group())
            if isinstance(parsed, list):
                # Extract citations from parsed list items
                for item in parsed:
                    item_str = str(item).strip()
                    # Extract citations from each item (may have descriptions in parens)
                    extracted = extract_citations(item_str)
                    citations.extend(extracted)
    except (ValueError, SyntaxError):
        pass
    
    # If no citations found from list parsing, try extracting from full output
    if not citations:
        citations = extract_citations(raw_output)

    return citations

In [16]:
# Test generation with a sample query
test_query = "What are the requirements for a valid contract under Swiss law?"
print(f"Query: {test_query}")
raw_citations = generate_citations(test_query)
print("\nGenerated citations:")
for c in raw_citations:
    print(f"  - {c}")

Query: What are the requirements for a valid contract under Swiss law?
Raw output: ["Art. 1 OR", "Art. 11 OR", "Art. 12"]


Generated citations:
  - Art. 1 OR
  - Art. 11 OR


## 4. Load Test Data

In [17]:
import pandas as pd

# Load queries from the configured query file
if not QUERY_FILE.exists():
    raise FileNotFoundError(f"Query file not found: {QUERY_FILE}")

test_df = pd.read_csv(QUERY_FILE)

print(f"Loaded {len(test_df)} queries from {QUERY_FILE}")
print(f"Columns: {list(test_df.columns)}")

if IS_VALIDATION_MODE and "gold_citations" in test_df.columns:
    print(f"Gold citations available for evaluation")

test_df.head()

Loaded 40 queries from /kaggle/input/competitions/llm-agentic-legal-information-retrieval/test.csv
Columns: ['query_id', 'query']


,query_id,query
0,test_001,Four U.S.-based software companies (NorthWave ...
1,test_002,On 9 August 2011 a 62‑year‑old cyclist (the cl...
2,test_003,"On 12 March 2012, Meridian Leasing Ltd and Ori..."
3,test_004,"A publicly listed manufacturing company, Orion..."
4,test_005,"A logistics company (R Ltd.), which owns a dis..."


## 5. Generate Predictions

In [18]:
from tqdm.notebook import tqdm

# Generate predictions
predictions = []

assert test_df is not None, "test_df must be loaded before generating predictions"

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Generating predictions"):
    query_id = row["query_id"]
    query_text = row["query"]

    # Generate citations using regex extraction (no external normalizer needed)
    raw_citations = generate_citations(query_text)

    # Use " " (single space) when no citations found, otherwise join with ";"
    predicted = ";".join(raw_citations) if raw_citations else " "

    predictions.append({
        "query_id": query_id,
        "predicted_citations": predicted,
    })

print(f"\nGenerated predictions for {len(predictions)} queries")

Generating predictions:   0%|          | 0/40 [00:00<?, ?it/s]

Raw output: This query relates to a specific legal dispute and does not directly involve Swiss laws or citations. However, if we assume that the dispute is being heard in a Swiss cantonal court and that the applicant companies are seeking interim relief under Swiss law, then the following citations might be relevant:

Raw output: ["BGE X Y Z E. N" (expected decision number will vary), "Art. 83 SVG", "Art. 59 Abs. 1 SVG"]

Raw output: [["Art. 95 OR (Consumer Contracts Act)", "Art. 12a LCC (Consumer Credit Act)", "Art. 13 LCC", "Art. 14 LCC", "Art. 15 LCC", "Art. 16 LCC", "Art. 267a CO", "BGE 141 III 118 E. 2.3", "BGE 145 II 153 E. 2.1"]

Raw output: [1] "Art. 166 Abs. 2 SchKG", "BGE X Y Z (forthcoming case law on the suspension of the peremptory period in the context of lifting-of-opposition proceedings)"
[2] "Art. 104 ZGB", "BGE X Y Z E. N (forthcoming case law on the annulment of bankruptcy based on an extract from the debt enforcement register)"

Raw output: ["BGE 141 II 345 E. 1.1 (

In [19]:
# Preview predictions
predictions_df = pd.DataFrame(predictions)
predictions_df.head(10)

,query_id,predicted_citations
0,test_001,
1,test_002,Art. 83 SVG;Art. 59 Abs. 1 SVG
2,test_003,BGE 141 III 118 E. 2.3;BGE 145 II 153 E. 2.1;A...
3,test_004,Art. 166 Abs. 2 SCHKG;Art. 104 ZGB
4,test_005,BGE 141 II 345 E. 1.1;Art. 122 OR;Art. 123 OR;...
5,test_006,
6,test_007,BGE 141 II 123 E. 3;BGE 138 II 387 E. 2.1;BGE ...
7,test_008,
8,test_009,
9,test_010,Art. 151 STGB;Art. 155 STGB;Art. 398 STPO


## 6. Create Submission

In [20]:
# Save submission
submission_path = OUTPUT_PATH / "submission_01_notebook.csv"
predictions_df.to_csv(submission_path, index=False)

print(f"Submission saved to: {submission_path}")
print(f"Total predictions: {len(predictions_df)}")

# Show sample
print("\nSample submission:")
print(predictions_df.head())

Submission saved to: /kaggle/working/submission_01_notebook.csv
Total predictions: 40

Sample submission:
   query_id                                predicted_citations
0  test_001                                                   
1  test_002                     Art. 83 SVG;Art. 59 Abs. 1 SVG
2  test_003  BGE 141 III 118 E. 2.3;BGE 145 II 153 E. 2.1;A...
3  test_004                 Art. 166 Abs. 2 SCHKG;Art. 104 ZGB
4  test_005  BGE 141 II 345 E. 1.1;Art. 122 OR;Art. 123 OR;...


## 7. Evaluation (Validation Set)

In [21]:
# === EVALUATION: Compare predictions against gold citations ===
# Only runs when DATASET_MODE = "val" (gold_citations available)

if IS_VALIDATION_MODE and "gold_citations" in test_df.columns:
    print("=" * 60)
    print("VALIDATION EVALUATION")
    print("=" * 60)
    
    # Merge predictions with gold
    eval_df = predictions_df.merge(
        test_df[["query_id", "query", "gold_citations"]],
        on="query_id",
        how="inner",
    )
    
    results = []
    for _, row in eval_df.iterrows():
        pred_set = set(row["predicted_citations"].split(";")) if row["predicted_citations"].strip() else set()
        gold_set = set(g.strip() for g in str(row["gold_citations"]).split(";") if g.strip())
        
        # Remove empty strings
        pred_set.discard("")
        pred_set.discard(" ")
        
        tp = pred_set & gold_set
        fp = pred_set - gold_set
        fn = gold_set - pred_set
        
        precision = len(tp) / len(pred_set) if pred_set else 0.0
        recall = len(tp) / len(gold_set) if gold_set else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        
        results.append({
            "query_id": row["query_id"],
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "tp": len(tp),
            "fp": len(fp),
            "fn": len(fn),
            "predicted_count": len(pred_set),
            "gold_count": len(gold_set),
        })
        
        # Per-query details
        print(f"\n{'─' * 50}")
        print(f"Query: {row['query'][:80]}...")
        print(f"  Predicted: {len(pred_set)} | Gold: {len(gold_set)} | TP: {len(tp)} | FP: {len(fp)} | FN: {len(fn)}")
        print(f"  P={precision:.3f}  R={recall:.3f}  F1={f1:.3f}")
        if tp:
            print(f"  ✓ TRUE POSITIVES: {sorted(tp)}")
        if fp and len(fp) <= 10:
            print(f"  ✗ FALSE POSITIVES: {sorted(fp)}")
        if fn and len(fn) <= 10:
            print(f"  ✗ FALSE NEGATIVES: {sorted(fn)}")
    
    results_df = pd.DataFrame(results)
    
    # Macro averages (= competition metric)
    macro_p = results_df["precision"].mean()
    macro_r = results_df["recall"].mean()
    macro_f1 = results_df["f1"].mean()
    
    print(f"\n{'=' * 60}")
    print(f"MACRO SCORES (competition metric)")
    print(f"{'=' * 60}")
    print(f"  Macro Precision: {macro_p:.4f}")
    print(f"  Macro Recall:    {macro_r:.4f}")
    print(f"  Macro F1:        {macro_f1:.4f}  ← THIS IS THE SCORE TO BEAT")
    print(f"  Queries:         {len(results_df)}")
    print(f"\nPer-query F1:")
    for _, r in results_df.iterrows():
        bar = "█" * int(r["f1"] * 40)
        print(f"  {r['query_id']}: F1={r['f1']:.3f} {bar}")
    
    print(f"\n{'=' * 60}")
    print(f"Total TP across all queries: {results_df['tp'].sum()}")
    print(f"Total FP across all queries: {results_df['fp'].sum()}")
    print(f"Total FN across all queries: {results_df['fn'].sum()}")
else:
    print("Skipping evaluation — no gold_citations column (test mode)")
    print("Upload submission.csv to Kaggle to get Public/Private LB scores")

Skipping evaluation — no gold_citations column (test mode)
Upload submission.csv to Kaggle to get Public/Private LB scores


## Summary

This baseline notebook demonstrates a simple direct generation approach:

1. **Prompt engineering**: We use a structured prompt that asks the LLM to generate Swiss legal citations in standard format.

2. **Citation normalization**: The generated citations are normalized to canonical form for consistent evaluation.

3. **Limitations**:
   - The LLM may hallucinate non-existent citations
   - No access to actual legal documents for verification
   - Relies entirely on the LLM's training data knowledge

For better results, see the **Agentic Retrieval Baseline** notebook which uses search tools to ground the generation in actual legal documents.